# Multivariate outlier detection using isolation forest and kNN
This code identifies and removes potential outliers in the imputed dataset using two different algorithms:
   - An `IsolationForest` is fit to the imputed dataset to flag anomalous points.  
   - Outlier labels (`1` = outlier, `0` = inlier) and decision scores (anomaly scores) are added to the DataFrame.  
   - All rows labeled as outliers are removed, producing a cleaned DataFrame (`cleaned_imputed_df_IF`)
   
   - A k-nearest-neighbors (k-NN) based outlier detector is fit on the same imputed data.  
   - Outlier flags and scores are added to a new DataFrame (`imputed_df_KNN`).  
   - Again, rows labeled as outliers are removed, yielding a second cleaned dataset (`cleaned_imputed_df_knn`).
   - The overlap of outliers flagged by both methods is computed.  
   - The overall agreement between the two label sets is printed as a percentage.

In [ ]:
# Remove the non-feature columns before performing Isolation Forest
exclude_cols = ["blueWins", "PCA1", "PCA2", "tSNE1", "tSNE2", "Cluster"]

imputed_df_IF = imputed_df_dirty.drop(columns=exclude_cols, errors="ignore").copy()

# Run IF once with arbitrary contamination to get the score distribution
clf_temp = IForest(contamination=0.05, n_estimators=200, random_state=42)
clf_temp.fit(imputed_df_IF)
scores_temp = clf_temp.decision_scores_  # higher = more abnormal in PyOD

# Compute IQR-based contamination fraction
q1, q3 = np.percentile(scores_temp, [25, 75])
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
iqr_fraction_IF = np.mean(scores_temp > upper_bound)
print(f"Estimated contamination (IQR method): {iqr_fraction_IF:.2%}")

# Rerun IF using its own IQR-based contamination fraction
clf = IForest(contamination=iqr_fraction_IF, n_estimators=200, random_state=42)
clf.fit(imputed_df_IF)
labels = clf.labels_
scores = clf.decision_scores_

# Add back results to the full DataFrame for later use
imputed_df_dirty["outlier_flag"] = labels
imputed_df_dirty["outlier_score"] = scores

# Keep only the inliers
cleaned_imputed_df_IF = imputed_df_dirty[imputed_df_dirty["outlier_flag"] == 0].copy()
print("Remaining rows after removing outliers:", cleaned_imputed_df_IF.shape[0])

# Plot baseline score distribution
plt.figure(figsize=(6,3))
plt.hist(scores, bins=60, color="steelblue", edgecolor="k")
plt.title("IF score distribution")
plt.xlabel("Anomaly score")
plt.ylabel("Count")
plt.show()

# Plot IF score distribution with IQR cutoff
plt.figure(figsize=(6,3))
plt.hist(scores, bins=60, color="steelblue", edgecolor="k")
t = np.percentile(scores, 100*(1 - iqr_fraction_IF))
plt.axvline(t, color="red", linestyle="--", linewidth=2,
            label=f"{iqr_fraction_IF:.2%} cutoff")
plt.title("Isolation Forest score distribution")
plt.xlabel("IF anomaly score")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
##kNN section
# Remove non-feature columns
exclude_cols = ["blueWins", "PCA1", "PCA2", "tSNE1", "tSNE2", 
                "Cluster","outlier_flag", "outlier_score"]
imputed_df_IF = imputed_df_dirty.drop(columns=exclude_cols, errors="ignore").copy()

# Run kNN once to get baseline score distribution (same as with IF)
knn = KNN(n_neighbors=5, contamination=0.05, method='largest')
knn.fit(imputed_df_IF)
scores_temp = knn.decision_scores_

# Compute IQR-based contamination based on kNN
q1, q3 = np.percentile(scores_temp, [25, 75])
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
iqr_fraction = np.mean(scores_temp > upper_bound)
print(f"Estimated contamination (IQR method): {iqr_fraction:.2%}")

# Rerun kNN using its own contamination fraction
knn = KNN(n_neighbors=5, contamination=iqr_fraction, method='largest')
knn.fit(imputed_df_IF)
knn_labels = knn.labels_
knn_scores = knn.decision_scores_

# Add kNN results to the imputed_df as new columns
imputed_df_dirty['knn_outlier_flag'] = knn_labels
imputed_df_dirty['knn_outlier_score'] = knn_scores

print("kNN - number of outliers:", (knn_labels == 1).sum())

# Keep only inliers (label == 0)
cleaned_imputed_df_knn = imputed_df_dirty[imputed_df_dirty['knn_outlier_flag'] == 0].copy()

# Plot kNN score distributions
plt.hist(knn_scores, bins=60, color="orange", edgecolor="k")
t = np.percentile(knn_scores, 100*(1-iqr_fraction))
plt.axvline(t, linestyle="--", label=f"{iqr_fraction*100:.2f}% cutoff")
plt.title("kNN score distribution")
plt.xlabel("kNN anomaly score (neighbor distance)")
plt.ylabel("Count")
plt.legend()
plt.show()

# Log scale plot
plt.hist(knn_scores, bins=60, color="orange", edgecolor="k")
t = np.percentile(knn_scores, 100*(1-iqr_fraction))
plt.axvline(t, linestyle="--", label=f"{iqr_fraction*100:.2f}% cutoff")
plt.xscale("log")
plt.title("kNN score distribution (log scale)")
plt.xlabel("kNN anomaly score (log scale)")
plt.ylabel("Count")
plt.legend()
plt.show()

# Zoomed-in plot
plt.hist(knn_scores, bins=60, color="orange", edgecolor="k")
plt.xlim(0, np.percentile(knn_scores, 99))  # zoom into 99th percentile
t = np.percentile(knn_scores, 100*(1-iqr_fraction))
plt.axvline(t, linestyle="--", label=f"{iqr_fraction*100:.2f}% cutoff")
plt.title("kNN score distribution (zoomed)")
plt.xlabel("kNN anomaly score")
plt.ylabel("Count")
plt.legend()
plt.show()

# Compare overlap between IF and kNN methods
overlap = ((labels == 1) & (knn_labels == 1)).sum()
print("Overlap of IF + kNN outliers:", overlap)

# Agreement between IF and KNN methods
agreement = np.mean(labels == knn_labels)
print(f"Agreement between IF and kNN: {agreement:.2%}")